# บท 08 · ข้อมูลสตรีมและเวลาที่ใช้ตัดสินใจ

ข้อมูล DEMO สร้างเอง หน่วย USD สมมติ จุดเริ่มเวลา 2026-01-05 14:30 UTC ใช้ standard library และทำงานออฟไลน์ทั้งหมด รันจาก kernel ใหม่ตามลำดับ เป้าหมายคือแยกเวลาเหตุการณ์ออกจากเวลาที่ข้อมูลพร้อมใช้ ไม่ได้วัดผลตอบแทน

## 1. เหตุการณ์และลำดับที่ระบบรับ

Tick แต่ละตัวมี event time, receive time และ ID การรับข้อความเดิมซ้ำไม่ควรเพิ่ม volume ราคาเท่ากันเพียงอย่างเดียวไม่พอที่จะสรุปว่าเป็นข้อความซ้ำ

In [1]:
"""Offline event-time demonstration. No network or broker dependencies."""
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from statistics import mean
import json

BASE = datetime(2026, 1, 5, 14, 30, tzinfo=timezone.utc)

@dataclass(frozen=True)
class Tick:
    event_id: str
    event_s: int
    received_s: int
    price: float
    size: int = 1

TICKS = [
    Tick("t1", 10, 10, 100), Tick("t2", 40, 42, 101),
    Tick("t3", 70, 71, 102), Tick("t2", 40, 72, 101),
    Tick("t4", 65, 73, 101.5), Tick("t5", 55, 74, 99),
    Tick("t6", 125, 126, 103), Tick("t7", 190, 191, 104),
]


for tick in TICKS:
    print(tick)


Tick(event_id='t1', event_s=10, received_s=10, price=100, size=1)
Tick(event_id='t2', event_s=40, received_s=42, price=101, size=1)
Tick(event_id='t3', event_s=70, received_s=71, price=102, size=1)
Tick(event_id='t2', event_s=40, received_s=72, price=101, size=1)
Tick(event_id='t4', event_s=65, received_s=73, price=101.5, size=1)
Tick(event_id='t5', event_s=55, received_s=74, price=99, size=1)
Tick(event_id='t6', event_s=125, received_s=126, price=103, size=1)
Tick(event_id='t7', event_s=190, received_s=191, price=104, size=1)


## 2. ตัวสร้างแท่ง

ใช้แท่ง 60 วินาทีและรอข้อมูลช้า 5 วินาที ปิดแท่งเมื่อปลายแท่งไม่เกิน watermark เรียงภายในแท่งตาม event time ข้อมูลที่มาหลังปิดแท่งถูกกักไว้ตรวจ ส่วน ID ที่ payload ขัดกันทำให้เกิดข้อผิดพลาด

In [2]:
class EventTimeBars:
    """One synthetic instrument, 60-second bars and five-second lateness."""
    def __init__(self, bar_seconds=60, lateness_seconds=5):
        self.bar_seconds = bar_seconds
        self.lateness_seconds = lateness_seconds
        self.seen = {}
        self.pending = {}
        self.bars = []
        self.signals = []
        self.audit = []
        self.max_event_s = -1
        self.last_received_s = -1
        self.watermark_s = float("-inf")

    def ingest(self, tick):
        if tick.received_s < self.last_received_s:
            raise ValueError("Replay must preserve receive order")
        if tick.event_s > tick.received_s or tick.price <= 0 or tick.size <= 0:
            raise ValueError("Invalid synthetic tick")
        self.last_received_s = tick.received_s
        signature = (tick.event_s, tick.price, tick.size)
        if tick.event_id in self.seen:
            if self.seen[tick.event_id] != signature:
                raise ValueError("Conflicting payload for an existing event ID")
            self.audit.append((tick.event_id, "DUPLICATE"))
            return
        self.seen[tick.event_id] = signature
        start = tick.event_s // self.bar_seconds * self.bar_seconds
        if start + self.bar_seconds <= self.watermark_s:
            self.audit.append((tick.event_id, "LATE_QUARANTINE"))
            return
        self.pending.setdefault(start, []).append(tick)
        self.max_event_s = max(self.max_event_s, tick.event_s)
        self.watermark_s = self.max_event_s - self.lateness_seconds
        self.audit.append((tick.event_id, "ACCEPT"))
        for start in sorted(list(self.pending)):
            end = start + self.bar_seconds
            if end > self.watermark_s:
                continue
            rows = sorted(self.pending.pop(start), key=lambda t: (t.event_s, t.event_id))
            bar = dict(start_s=start, end_s=end, available_s=tick.received_s,
                       open=rows[0].price, high=max(t.price for t in rows),
                       low=min(t.price for t in rows), close=rows[-1].price,
                       volume=sum(t.size for t in rows))
            self.bars.append(bar)
            if len(self.bars) >= 3:
                recent = self.bars[-3:]
                if all(a["end_s"] == b["start_s"] for a, b in zip(recent, recent[1:])):
                    average = mean(b["close"] for b in recent)
                    self.signals.append(dict(bar_end_s=end, available_s=tick.received_s,
                                             sma3=average, target=int(bar["close"] > average)))


print("EventTimeBars is ready; no network connection")


EventTimeBars is ready; no network connection


## 3. Replay และตรวจผลแท่ง

Replay ตามลำดับที่รับเพื่อรักษาความจริงว่าโปรแกรมรู้อะไรในแต่ละจุด ไม่เรียงข้อมูลทั้งหมดตาม event time ล่วงหน้า

In [3]:
def replay(ticks=TICKS):
    engine = EventTimeBars()
    for tick in ticks:
        engine.ingest(tick)
    return engine

def freshness(engine, now_s, max_age_s=20):
    """Use the newest accepted event time; duplicates do not refresh price age."""
    age = now_s - engine.max_event_s
    return dict(age_seconds=age, allow_new_orders=0 <= age <= max_age_s)


engine = replay()
for bar in engine.bars:
    print(bar)
print("audit:", engine.audit)


{'start_s': 0, 'end_s': 60, 'available_s': 71, 'open': 100, 'high': 101, 'low': 100, 'close': 101, 'volume': 2}
{'start_s': 60, 'end_s': 120, 'available_s': 126, 'open': 101.5, 'high': 102, 'low': 101.5, 'close': 102, 'volume': 2}
{'start_s': 120, 'end_s': 180, 'available_s': 191, 'open': 103, 'high': 103, 'low': 103, 'close': 103, 'volume': 1}
audit: [('t1', 'ACCEPT'), ('t2', 'ACCEPT'), ('t3', 'ACCEPT'), ('t2', 'DUPLICATE'), ('t4', 'ACCEPT'), ('t5', 'LATE_QUARANTINE'), ('t6', 'ACCEPT'), ('t7', 'ACCEPT')]


## 4. เมื่อสัญญาณพร้อม และเมื่อข้อมูลเก่า

SMA ของราคาปิด 101, 102, 103 คือ 102 สัญญาณของแท่งที่จบวินาที 180 พร้อมที่ receive time 191 อย่าย้อนให้ fill เกิดก่อนสัญญาณพร้อม เกณฑ์ 20 วินาทีตรวจความสดของ tick; ในระบบจริงต้องตรวจอายุแท่งที่ใช้ด้วย

In [4]:
print("signals:", engine.signals)
for now in (192, 220):
    print(now, freshness(engine, now))
assert engine.signals[0]["available_s"] > engine.signals[0]["bar_end_s"]


signals: [{'bar_end_s': 180, 'available_s': 191, 'sma3': 102, 'target': 1}]
192 {'age_seconds': 2, 'allow_new_orders': True}
220 {'age_seconds': 30, 'allow_new_orders': False}


## 5. แบบฝึกหัดพร้อมเฉลย: รอ 15 วินาที

คาดผลก่อนรัน: เมื่อรอนานขึ้น t5 จะเข้าทันแท่งแรก ราคาปิดแท่งแรกจึงเป็น 99 และพร้อมวินาที 126 ทั้งข้อมูลและเวลาตัดสินใจเปลี่ยน

In [5]:
patient = EventTimeBars(lateness_seconds=15)
for tick in TICKS:
    patient.ingest(tick)
for bar in patient.bars:
    print(bar)
assert patient.bars[0]["close"] == 99
assert patient.bars[0]["available_s"] == 126
assert patient.bars[0]["volume"] == 3


{'start_s': 0, 'end_s': 60, 'available_s': 126, 'open': 100, 'high': 101, 'low': 99, 'close': 99, 'volume': 3}
{'start_s': 60, 'end_s': 120, 'available_s': 191, 'open': 101.5, 'high': 102, 'low': 101.5, 'close': 102, 'volume': 2}


## 6. ตรวจพฤติกรรมสำคัญ

เช็กการเรียง ข้อมูลซ้ำ การกักข้อมูลมาช้า เวลาสัญญาณ ความสด และความขัดแย้งของ payload ผล PASS บอกว่า logic นี้ตรงกับสมมติฐานที่ตั้งไว้ ไม่ยืนยันความครบถ้วนของฟีดจริง

In [6]:
def checks():
    engine = replay()
    assert [b["close"] for b in engine.bars] == [101, 102, 103]
    assert engine.bars[1]["open"] == 101.5
    assert [b["volume"] for b in engine.bars] == [2, 2, 1]
    assert engine.signals == [dict(bar_end_s=180, available_s=191, sma3=102, target=1)]
    assert sum(state == "DUPLICATE" for _, state in engine.audit) == 1
    assert sum(state == "LATE_QUARANTINE" for _, state in engine.audit) == 1
    assert len(engine.pending) == 1
    assert freshness(engine, 192)["allow_new_orders"]
    assert not freshness(engine, 220)["allow_new_orders"]
    assert replay().bars == engine.bars
    try:
        engine.ingest(Tick("t2", 40, 192, 999))
    except ValueError:
        pass
    else:
        raise AssertionError("Conflicting duplicate must fail")
    return "10 event-time checks passed"


print(checks())


10 event-time checks passed


## อ่านต่อ

Hilpisch บท 7 หน้าในเล่ม 201, 208 (PDF 221, 228) เป็นกรอบการประมวลผลออนไลน์; ตัวอย่างและ watermark เขียนใหม่สำหรับหลักสูตร

[Webull Thailand Streaming](https://developer.webull.co.th/apis/docs/reference/trade-api/market-data-streaming/) ตรวจ 11 กันยายน 2026. Notebook ไม่ได้ทดสอบการเชื่อมต่อหรือความพร้อมของสิทธิ์ข้อมูล